# Setup CBAM-ResNet18 di Google Colab

Aktifkan **Runtime > Change runtime type > GPU**, lalu jalankan sel secara berurutan.
Notebook menyiapkan environment proyek dan satu langkah forward/backward GPU.
Pelatihan 30 epoch tidak berjalan otomatis. Model dan loader berasal dari modul repositori.

Gunakan URL HTTPS repositori GitHub yang dapat diakses. Data dan environment diunduh
ke runtime Colab; tidak ada penghubungan Google Drive otomatis.

In [ ]:
import importlib.metadata
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
from urllib.parse import urlparse

if not Path('/content').is_dir() or sys.platform != 'linux':
    raise RuntimeError('Notebook setup ini ditujukan untuk runtime Google Colab Linux.')
if not (3, 12) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(f'Gunakan runtime Python 3.12 atau 3.13; saat ini {sys.version}.')
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('GPU NVIDIA belum tersedia. Pilih runtime GPU, lalu ulangi.')
subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total',
                '--format=csv,noheader'], check=True)
pip_version = importlib.metadata.version('pip')
if tuple(map(int, pip_version.split('.')[:2])) < (22, 3):
    raise RuntimeError('Perbarui pip runtime ke versi >=22.3 sebelum menjalankan setup.')
print(f'Python runtime: {sys.version.split()[0]}; pip: {pip_version}')


In [ ]:
REPO_DIR = Path('/content/cbam-resnet18-imagenette-160')
if not REPO_DIR.exists():
    repo_url = input('URL HTTPS repositori GitHub: ').strip()
    parsed = urlparse(repo_url)
    if (parsed.scheme != 'https' or parsed.hostname != 'github.com'
            or parsed.username or parsed.password or parsed.query or parsed.fragment
            or len(parsed.path.strip('/').split('/')) != 2):
        raise ValueError('Masukkan URL repositori, misalnya https://github.com/pemilik/nama-repo.git, tanpa token.')
    subprocess.run(['git', 'clone', '--', repo_url, str(REPO_DIR)], check=True)
for relative in ('train.py', 'check_env.py', 'requirements/common.txt', 'requirements/pytorch-cu118.txt'):
    if not (REPO_DIR / relative).is_file():
        raise RuntimeError(f'Berkas proyek tidak ditemukan: {REPO_DIR / relative}')
os.chdir(REPO_DIR)
print(f'Root proyek: {REPO_DIR}')


In [ ]:
# Venv terpisah menjaga paket bawaan notebook Colab tetap terpisah dari proyek.
VENV_DIR = REPO_DIR / '.venv'
COLAB_PYTHON = VENV_DIR / 'bin' / 'python'
if not COLAB_PYTHON.is_file():
    subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(VENV_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', '--python', str(COLAB_PYTHON),
                'install', 'pip==25.3'], check=True)
for requirements in ('requirements/pytorch-cu118.txt', 'requirements/common.txt'):
    subprocess.run([str(COLAB_PYTHON), '-m', 'pip', 'install', '--no-cache-dir',
                    '-r', requirements], cwd=REPO_DIR, check=True)
subprocess.run([str(COLAB_PYTHON), '-m', 'pip', 'check'], check=True)


In [ ]:
# Data lengkap yang sudah ada dipakai kembali oleh pengunduh.
subprocess.run([str(COLAB_PYTHON), '-m', 'utils.download_data'], cwd=REPO_DIR, check=True)


In [ ]:
REPORT_PATH = REPO_DIR / 'outputs' / 'environment-colab.json'
subprocess.run([str(COLAB_PYTHON), 'check_env.py', '--require-cuda',
                '--output', str(REPORT_PATH)], cwd=REPO_DIR, check=True)
report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
print(json.dumps({key: report[key] for key in ('status', 'device', 'dataset', 'smoke_test')},
                 indent=2, ensure_ascii=False))


## Setelah setup berhasil

Laporan lingkungan tersimpan di `outputs/environment-colab.json`. Simpan atau unduh
laporan tersebut sebelum runtime dihapus. Waktu dan memori yang tercatat adalah
hasil batch pemeriksaan kecil, bukan hasil eksperimen penuh.

Untuk menguji pipeline satu epoch, buat sel baru dan jalankan:

```python
subprocess.run([str(COLAB_PYTHON), 'train.py', '--epochs', '1', '--seeds', '0',
                '--device', 'cuda', '--run-name', 'smoke-1epoch'], cwd=REPO_DIR, check=True)
```

Hasil berada di `outputs/smoke-1epoch/`, termasuk checkpoint, CSV, metrik, dan grafik.

Untuk eksperimen utama 30 epoch per seed (0, 1, 2), jalankan sel terpisah:

```python
subprocess.run([str(COLAB_PYTHON), 'train.py', '--device', 'cuda',
                '--run-name', 'cbam-full-30epoch'], cwd=REPO_DIR, check=True)
```

Gunakan nama run baru untuk percobaan berikutnya. Simpan seluruh folder hasil
sebelum runtime dihapus. Uji satu epoch belum menggantikan eksperimen utama.